# 🌍 Carbon Footprint Predictor

**Predict an individual's annual CO₂ emissions (kg) based on daily lifestyle factors.**

This project uses a modular scikit-learn Pipeline with ColumnTransformer to preprocess features and compare two regression models:
- Linear Regression (baseline)
- Random Forest Regressor

**Dataset:** [Carbon Footprint Dataset — Kaggle](https://www.kaggle.com/datasets/dumanmesut/individual-carbon-footprint-calculation)

**Author:** Sai Ramya
**GSSoC 2026 Contribution**

## 📦 Install Dependencies

In [ ]:
!pip install pandas numpy scikit-learn matplotlib seaborn --quiet

## 📥 Load Dataset

Dataset source: [Kaggle — Individual Carbon Footprint Calculation](https://www.kaggle.com/datasets/dumanmesut/individual-carbon-footprint-calculation)

Download the dataset from Kaggle and place `carbon_footprint_dataset.csv` in the same folder as this notebook.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('carbon_footprint_dataset.csv')
print(f'Dataset shape: {df.shape}')
df.head()

## 🔍 Data Exploration

In [ ]:
print('--- Dataset Info ---')
df.info()
print('\n--- Statistical Summary ---')
df.describe()

In [ ]:
print('--- Missing Values ---')
print(df.isnull().sum())
print(f'\nMean CO2 emissions: {df["CarbonEmission"].mean():.2f} kg')
print(f'Min: {df["CarbonEmission"].min():.2f} kg')
print(f'Max: {df["CarbonEmission"].max():.2f} kg')

## 📊 Data Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['CarbonEmission'], bins=40, color='#6366f1', edgecolor='white', alpha=0.85)
axes[0].set_title('Distribution of Carbon Emissions', fontsize=13, fontweight='bold')
axes[0].set_xlabel('CO2 Emissions (kg)')
axes[0].set_ylabel('Frequency')

if 'Transport' in df.columns:
    transport_means = df.groupby('Transport')['CarbonEmission'].mean().sort_values(ascending=False)
    axes[1].bar(transport_means.index, transport_means.values, color='#22d3ee', edgecolor='white', alpha=0.85)
    axes[1].set_title('Avg Emissions by Transport Type', fontsize=13, fontweight='bold')
    axes[1].set_xlabel('Transport')
    axes[1].set_ylabel('Avg CO2 (kg)')
    axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
plt.figure(figsize=(12, 8))
sns.heatmap(
    df[num_cols].corr(),
    annot=True, fmt='.2f',
    cmap='coolwarm',
    linewidths=0.5,
    annot_kws={'size': 8}
)
plt.title('Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## ⚙️ Data Preprocessing

Using `ColumnTransformer` to apply:
- `OneHotEncoder` on categorical features
- `StandardScaler` on numerical features

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

X = df.drop('CarbonEmission', axis=1)
y = df['CarbonEmission']

cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = X.select_dtypes(include=np.number).columns.tolist()

print(f'Categorical features ({len(cat_cols)}): {cat_cols}')
print(f'Numerical features  ({len(num_cols)}): {num_cols}')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'\nTrain size: {X_train.shape[0]} | Test size: {X_test.shape[0]}')

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

## 📈 Model 1: Linear Regression (Baseline)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)

lr_mae = mean_absolute_error(y_test, y_pred_lr)
lr_r2  = r2_score(y_test, y_pred_lr)

print('Linear Regression Results')
print(f'  MAE : {lr_mae:.2f} kg')
print(f'  R2  : {lr_r2:.4f}')

## 🌲 Model 2: Random Forest Regressor

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42))
])

rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)

rf_mae = mean_absolute_error(y_test, y_pred_rf)
rf_r2  = r2_score(y_test, y_pred_rf)

print('Random Forest Results')
print(f'  MAE : {rf_mae:.2f} kg')
print(f'  R2  : {rf_r2:.4f}')

## 📊 Model Comparison

In [ ]:
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest'],
    'MAE':   [round(lr_mae, 2), round(rf_mae, 2)],
    'R2':    [round(lr_r2, 4),  round(rf_r2, 4)]
})
print(results.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(results['Model'], results['MAE'], color=['#6366f1', '#22d3ee'], edgecolor='white')
axes[0].set_title('MAE Comparison (lower is better)', fontweight='bold')
axes[0].set_ylabel('MAE (kg)')

axes[1].bar(results['Model'], results['R2'], color=['#6366f1', '#22d3ee'], edgecolor='white')
axes[1].set_title('R2 Score Comparison (higher is better)', fontweight='bold')
axes[1].set_ylabel('R2 Score')
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

## 🔍 Prediction vs Actual

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_rf, alpha=0.4, color='#6366f1', edgecolors='white', linewidth=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2, label='Perfect Prediction')
plt.xlabel('Actual CO2 Emissions (kg)')
plt.ylabel('Predicted CO2 Emissions (kg)')
plt.title('Random Forest: Actual vs Predicted', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

## ✅ Conclusion

| Model | MAE | R2 |
|---|---|---|
| Linear Regression | 285.19 kg | 0.89 |
| Random Forest | 304.29 kg | 0.90 |

**Key Takeaways:**
- Both models achieve high R2 scores (>0.89)
- Random Forest achieves slightly better R2 but higher MAE
- Linear Regression is a strong baseline for this structured dataset
- The sklearn Pipeline ensures clean, reproducible preprocessing

**Future Improvements:**
- Hyperparameter tuning with GridSearchCV
- Try XGBoost or Gradient Boosting
- Deploy as a Streamlit web app